# CNN implementation and threshold-transfer audit

Exploratory, post-evaluation diagnostics of the frozen run. No training, threshold replacement, or model selection. Original artifacts remain unchanged. Use the project environment. Generated diagnostics go into a fresh timestamped reports directory. Subject identities remain unverified; group associations are not causal explanations.

**Showcase scope:** this rerunnable copy replays the five distributed primary CNNs. The original executed snapshot records the original 15-model audit; extra-seed model binaries are intentionally not distributed. All reported seed predictions and metrics remain included.

The distributed repository omits the redundant waveform memory-map cache. When absent, this audit reconstructs a temporary input array from the exact prepared waveform release. The original executed snapshot records the original cache comparison; this rerun checks reconstruction and primary-model replay.

In [1]:
import ast
import hashlib
import json
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve, roc_auc_score
from torch import nn  # noqa: F401 -- used by frozen AST definition

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RUN = ROOT / "data/processed/portfolio-v2/plan001-real-20261108"
FULL = RUN / "full/real-20261109-v1"
R = FULL / "results"
OUT = ROOT / "reports/portfolio-v2/cnn-audit" / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
OUT.mkdir(parents=True, exist_ok=False)
torch.set_num_threads(4)
manifest = json.loads((R / "manifest.json").read_text())
for name, expected in manifest["artifacts"].items():
    assert hashlib.sha256((R / name).read_bytes()).hexdigest() == expected, name
freeze = json.loads((RUN / "freeze/payload.json").read_text())
for name, expected in freeze["identity"]["source_hashes"].items():
    assert hashlib.sha256((ROOT / "notebooks" / name).read_bytes()).hexdigest() == expected
assert (
    hashlib.sha256((ROOT / "notebooks/08_nested_evaluation.ipynb").read_bytes()).hexdigest()
    == manifest["context"]["runner_sha256"]
)
cohort = pd.read_parquet(RUN / "protocol/cohort.parquet").reset_index(drop=True)
pred = pd.read_parquet(R / "outer_predictions.parquet")
oof = pd.read_parquet(R / "inner_oof_predictions.parquet")
choices = pd.DataFrame(json.loads((R / "selected_configurations.json").read_text()))
history = pd.read_parquet(R / "training_history.parquet")
cache = FULL / "inputs/waveforms.npy"
if not cache.exists():
    # The redundant original cache is omitted from the showcase.
    # Reconstruct a temporary audit input from the exact released prepared arrays.
    cache = OUT / "audit-waveforms.npy"
    array = np.lib.format.open_memmap(cache, mode="w+", dtype="float32", shape=(len(cohort), 6000))
    for record, part in cohort.groupby("record_id"):
        with np.load(ROOT / "data/processed/phase1/waveforms" / f"{record}.npz") as source:
            signal = source["filtered"]
        for row in part.itertuples():
            values = np.asarray(signal[row.start_sample : row.end_sample], dtype=float)
            array[row.Index] = np.clip(
                (values - values.mean()) / max(float(values.std()), 0.001), -8, 8
            ).astype("float32")
    array.flush()
    del array
waves = np.load(cache, mmap_mode="r")
assert waves.shape == (len(cohort), 6000) and waves.dtype == np.float32
print("Provenance verified. Output:", OUT)

Provenance verified. Output: <project-root>/reports/portfolio-v2/cnn-audit/20261110T043659174717Z


## Independently reconstruct every cached minute

Check source waveform hashes, 100 Hz sampling, exact minute boundaries, ordering, finite values, and normalization. Recompute normalization directly from filtered Phase 1 samples rather than calling the runner. This detects cache/input misalignment but does not independently validate filtering physiology or the original annotations.

In [2]:
quality = []
for record, part in cohort.groupby("record_id"):
    path = ROOT / "data/processed/phase1/waveforms" / f"{record}.npz"
    expected = freeze["identity"]["input_hashes"][f"waveforms/{record}.npz"]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected
    with np.load(path) as source:
        assert int(source["fs"]) == 100
        signal = source["filtered"]
        for row in part.itertuples():
            assert row.end_sample - row.start_sample == 6000
            assert row.start_sample == row.minute * 6000
            raw = np.asarray(signal[row.start_sample : row.end_sample], dtype=float)
            assert raw.shape == (6000,) and np.isfinite(raw).all()
            normalized = np.clip((raw - raw.mean()) / max(raw.std(), 0.001), -8, 8).astype(
                "float32"
            )
            np.testing.assert_array_equal(normalized, waves[row.Index])
            quality.append(
                dict(
                    record_id=record,
                    minute=row.minute,
                    subject_group=row.subject_group,
                    label=row.label,
                    raw_std=raw.std(),
                    normalized_std=normalized.std(),
                    clipped_fraction=np.mean(np.abs(normalized) >= 8),
                )
            )
quality = pd.DataFrame(quality)
display(quality[["raw_std", "normalized_std", "clipped_fraction"]].describe())
print("Exact cache reconstruction passed for", len(quality), "minutes.")

,raw_std,normalized_std,clipped_fraction
count,1.655800e+04,16558.000000,16558.000000
mean,2.983590e-01,0.997641,0.000146
std,1.866918e-01,0.032835,0.000512
min,2.200679e-62,0.000000,0.000000
25%,1.656621e-01,1.000000,0.000000
50%,2.559888e-01,1.000000,0.000000
75%,3.984697e-01,1.000000,0.000000
max,4.576155e+00,1.000000,0.006833


Exact cache reconstruction passed for 16558 minutes.


## Replay saved CNNs without the runner inference function

Load only the frozen architecture definition. Independently apply each exported state dictionary, sigmoid once, and compare all saved outer scores (published primary seed). This checks model export, sample ordering, logits/probability conversion, and inference consistency. It does not show that training learned a generalizable representation.

In [3]:
notebook = json.loads((ROOT / "notebooks/07_nested_model_comparison.ipynb").read_text())
for cell in notebook["cells"]:
    if cell["cell_type"] == "code":
        for node in ast.parse("".join(cell["source"])).body:
            if isinstance(node, ast.FunctionDef) and node.name == "make_cnn":
                exec(
                    compile(
                        ast.Module(body=[node], type_ignores=[]), "<frozen-architecture>", "exec"
                    )
                )
make_cnn = globals()["make_cnn"]
replays = []
lookup = cohort.reset_index().set_index(["record_id", "minute"])["index"]
for (fold, seed), part in pred.query(
    "analysis == 'tuned' and model == 'cnn' and seed == 67"
).groupby(["outer_fold", "seed"]):
    model = make_cnn()
    assert sum(p.numel() for p in model.parameters()) == 3009
    path = R / "models" / f"outer-{fold}__tuned__cnn__seed-{seed}__outer.pt"
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    model.eval()
    indices = lookup.loc[pd.MultiIndex.from_frame(part[["record_id", "minute"]])].to_numpy()
    scores = []
    with torch.inference_mode():
        for start in range(0, len(indices), 128):
            x = torch.tensor(
                np.array(waves[indices[start : start + 128], None, :]), dtype=torch.float32
            )
            scores.extend(torch.sigmoid(model(x).squeeze(1)).tolist())
    error = np.max(np.abs(np.asarray(scores) - part.score.to_numpy()))
    assert error < 1e-6
    replays.append(dict(outer_fold=int(fold), seed=int(seed), max_score_error=float(error)))
replays = pd.DataFrame(replays)
display(replays)

,outer_fold,seed,max_score_error
0,0,67,0.0
1,1,67,0.0
2,2,67,0.0
3,3,67,0.0
4,4,67,0.0


## Independently verify thresholds and compare inner versus outer distributions

Recompute maximum inner OOF F1 using sklearn's precision–recall thresholds, including the frozen zero candidate and lowest-threshold tie rule. Compare the selected threshold with class-specific score distributions on held-out data. A zero threshold makes any nonnegative probability score positive by construction. Positive rates near one on inner OOF data indicate a permissive operating point was already selected, not solely an outer-distribution failure.

In [4]:
rows = []
for (model, fold), inner in oof.query("analysis == 'tuned' and seed == 67").groupby(
    ["model", "outer_fold"]
):
    outer = pred.query("analysis == 'tuned' and seed == 67")
    outer = outer[outer.model.eq(model) & outer.outer_fold.eq(fold)]
    precision, recall, thresholds = precision_recall_curve(inner.label, inner.score)
    f1 = np.divide(
        2 * precision[:-1] * recall[:-1],
        precision[:-1] + recall[:-1],
        out=np.zeros(len(thresholds)),
        where=(precision[:-1] + recall[:-1]) > 0,
    )
    thresholds = np.r_[0.0, thresholds]
    f1 = np.r_[f1_score(inner.label, np.ones(len(inner))), f1]
    best = thresholds[np.isclose(f1, f1.max(), rtol=0, atol=1e-12)].min()
    saved = float(inner.threshold.iloc[0])
    assert np.isclose(best, saved, rtol=0, atol=1e-12), (model, fold, best, saved)
    row = dict(
        model=model,
        fold=int(fold) + 1,
        threshold=saved,
        inner_positive_rate=float(inner.prediction.mean()),
        outer_positive_rate=float(outer.prediction.mean()),
        inner_f1=float(f1_score(inner.label, inner.prediction)),
        outer_f1=float(f1_score(outer.label, outer.prediction)),
    )
    for name, part in [("inner", inner), ("outer", outer)]:
        for label in [0, 1]:
            values = part.loc[part.label.eq(label), "score"]
            row[f"{name}_class{label}_median"] = float(values.median())
            row[f"{name}_class{label}_min"] = float(values.min())
    rows.append(row)
transfer = pd.DataFrame(rows)
display(transfer.query("model == 'cnn'"))

,model,fold,threshold,inner_positive_rate,outer_positive_rate,inner_f1,outer_f1,inner_class0_median,inner_class0_min,inner_class1_median,inner_class1_min,outer_class0_median,outer_class0_min,outer_class1_median,outer_class1_min
0,cnn,1,0.424929,0.594673,0.758060,0.663601,0.589624,0.378184,0.070039,0.602280,0.068254,0.629020,0.171164,0.674726,0.204767
1,cnn,2,0.286781,0.507931,0.407255,0.661196,0.534505,0.171909,0.000026,0.478610,0.000160,0.155672,0.004487,0.349770,0.010464
2,cnn,3,0.368988,0.516523,0.267239,0.647346,0.196589,0.356172,0.007598,0.462133,0.020904,0.266678,0.034923,0.243643,0.028958
3,cnn,4,0.246968,0.999773,1.000000,0.562796,0.570516,0.378027,0.233971,0.374288,0.246968,0.385151,0.344235,0.391691,0.366200
4,cnn,5,0.240311,0.855281,1.000000,0.576688,0.578561,0.367648,0.001959,0.372102,0.028646,0.372340,0.352865,0.375707,0.356702


## Group and training diagnostics

Describe group AP relative to its own prevalence, and signal scale/clipping by group. Single-class group AP/AUC is omitted. These associations do not establish the cause of poor performance. Training loss and unweighted stopping loss have different weighting and need not move together. Inspect selected epoch schedules rather than choosing new ones from outer performance.

In [5]:
group_rows = []
for (model, group), part in pred.query("analysis == 'tuned' and seed == 67").groupby(
    ["model", "subject_group"]
):
    both = part.label.nunique() == 2
    ap = average_precision_score(part.label, part.score) if both else np.nan
    group_rows.append(
        dict(
            model=model,
            subject_group=group,
            n=len(part),
            prevalence=part.label.mean(),
            ap=ap,
            ap_above_prevalence=ap - part.label.mean(),
            auc=roc_auc_score(part.label, part.score) if both else np.nan,
            positive_rate=part.prediction.mean(),
        )
    )
groups = pd.DataFrame(group_rows).merge(
    quality.groupby("subject_group").agg(
        raw_std_median=("raw_std", "median"), clipped_mean=("clipped_fraction", "mean")
    ),
    on="subject_group",
    validate="many_to_one",
)
display(groups.query("model == 'cnn'").sort_values("ap_above_prevalence"))
selected = choices.query("analysis == 'tuned' and model == 'cnn' and seed == 67")
display(selected[["outer_fold", "candidate", "settings", "calibrated_epochs", "outer_epochs"]])
assert np.isfinite(history.train_loss).all()
assert np.isfinite(history.loc[history.stage.eq("calibration"), "stop_loss"]).all()
for name, table in {
    "signal_quality": quality,
    "cnn_replay": replays,
    "threshold_transfer": transfer,
    "group_metrics": groups,
    "cnn_selected_settings": selected,
}.items():
    table.to_csv(OUT / f"{name}.csv", index=False)
print("Audit checks passed; these checks do not establish absence of every possible defect.")

,model,subject_group,n,prevalence,ap,ap_above_prevalence,auc,positive_rate,raw_std_median,clipped_mean
1,cnn,subj_a02,528,0.795455,0.688453,-0.107002,0.252557,0.547348,0.118462,3.219697e-05
10,cnn,subj_a15,509,0.722986,0.628268,-0.094718,0.371049,0.785855,0.499578,9.233792e-05
7,cnn,subj_a08,994,0.434608,0.364107,-0.070501,0.398239,1.000000,0.603649,3.353454e-06
3,cnn,subj_a04,1069,0.923293,0.866709,-0.056584,0.409457,0.087933,0.122617,8.217961e-04
9,cnn,subj_a11,466,0.476395,0.442992,-0.033403,0.294085,1.000000,0.284851,2.861230e-06
16,cnn,subj_b05,432,0.131944,0.120512,-0.011433,0.498175,1.000000,0.351263,1.270448e-03
15,cnn,subj_b04,941,0.010627,0.006725,-0.003902,0.189366,0.068013,0.111978,7.545165e-05
6,cnn,subj_a07,991,0.646821,0.643950,-0.002871,0.514005,0.995964,0.566697,5.045409e-07
18,cnn,subj_c02,968,0.003099,0.003087,-0.000012,0.346805,0.539256,0.328318,2.858127e-05
21,cnn,subj_c06,467,0.002141,0.003185,0.001043,0.328326,1.000000,0.173796,2.997859e-05


,outer_fold,candidate,settings,calibrated_epochs,outer_epochs
8,0,11.0,"{'batch_size': 128, 'learning_rate': 0.0029448...","[1, 1, 2]",1.0
34,1,5.0,"{'batch_size': 32, 'learning_rate': 0.00141735...","[1, 12, 1]",1.0
60,2,2.0,"{'batch_size': 128, 'learning_rate': 0.0010115...","[4, 3, 1]",3.0
86,3,13.0,"{'batch_size': 64, 'learning_rate': 0.00032407...","[2, 5, 2]",2.0
112,4,14.0,"{'batch_size': 64, 'learning_rate': 0.00020179...","[23, 1, 1]",1.0


Audit checks passed; these checks do not establish absence of every possible defect.


## Audit conclusion

Exact reconstruction and model replay passed; no cache-ordering or inference-conversion defect was established. However, folds 4/5 already selected very permissive inner thresholds, and primary outer CNN refits used just 1–3 epochs. Thus all-positive predictions are not solely an unseen-group score shift. There are 17 near-flat filtered minutes requiring upstream review. See `docs/cnn-implementation-audit.md` for evidence, interpretation, and the remaining checks. These are exploratory leads, not proven causes or authorization to revise the frozen scores.